# 🗃️ Introduction to Databricks SQL — Project Showcase

![Databricks](https://img.shields.io/badge/Platform-Databricks-FF3621?logo=databricks&logoColor=white)
![SQL](https://img.shields.io/badge/Language-SQL-4479A1?logo=postgresql&logoColor=white)
![Dataset](https://img.shields.io/badge/Dataset-Insurance%20Claims-blueviolet)
![Status](https://img.shields.io/badge/Status-Completed-brightgreen)

**Role I played in this project:** data analyst supporting an insurance company's operations team, working directly inside the Databricks SQL Editor and Dashboards against a live `insurance` table in Unity Catalog.

**What this notebook demonstrates:** the ability to go from a raw table to a governed, multi-layer analytics asset — writing ANSI SQL queries, cleaning and joining data through a Bronze → Silver → Gold pipeline, building visualizations, and assembling interactive dashboards that a business stakeholder (e.g. a VP of Claims, or an executive team) could actually use to make decisions.

All queries below are the ones I wrote and ran in the Databricks SQL Editor; the screenshots are the actual results returned by the platform.

---


## Chapter 1 — Querying and Visualizing Insurance Data

The first chapter got me comfortable with the core building blocks of Databricks SQL: the **SQL Editor**, **Unity Catalog**, and the relationship between **Queries → Tables/Views → Visualizations → Dashboards**. I used a real `insurance` table (claims data with fields like `HOUSE_TYPE`, `RISK_SEGMENTATION`, `PREMIUM_AMOUNT`, and `STATE`) to answer concrete business questions.


### 1.1 — Exploring the insurance dataset

I started by previewing the `insurance` table directly from the Catalog, then moved into the SQL Editor to answer a specific question from the business: **how many claims come from rented properties classified as high risk?** I filtered on `HOUSE_TYPE = 'Rent'` and `RISK_SEGMENTATION = 'H'`.

```sql
SELECT *
FROM `databricks_ws_...`.`default`.`insurance`
WHERE HOUSE_TYPE = 'Rent'
  AND RISK_SEGMENTATION = 'H';
```

**Result: 505 claims** met both conditions — a useful starting point for a risk-review conversation with underwriting.

![Filtering the insurance table for rented, high-risk claims](./screenshots/image1.png)


### 1.2 — Connecting to a BI tool with Partner Connect

Beyond the native SQL Editor, Databricks SQL Warehouses can be plugged directly into external BI tools. I used **Partner Connect** to generate a connection file for **Power BI Desktop**, choosing a Serverless Starter Warehouse as the compute backend. This creates a governed Power BI dataset that queries the warehouse directly — meaning downstream BI users get Databricks' scalability, governance (Unity Catalog), and performance without leaving their tool of choice.

![Setting up a Power BI connection through Partner Connect](./screenshots/image2.png)


### 1.3 — Building a reusable query: average premium by state

Next I built a named, reusable query (`Average_Premiums`) rather than a one-off `SELECT`, since this is a metric the business would want to revisit regularly.

```sql
SELECT STATE, ROUND(AVG(PREMIUM_AMOUNT), 2) AS avg_premium
FROM `databricks_ws_...`.`default`.`insurance`
GROUP BY STATE;
```

**Result:** the average premium for a customer in **Arizona is $89.47**.

![Average premium query by state](./screenshots/image3.png)


### 1.4 — Turning the query into a visualization

A table of 50 states is hard to scan for outliers, so I built a **horizontal bar chart** (`avg(PREMIUM_AMOUNT)` on X, `STATE` on Y, data labels on) to directly answer a colleague's question: *"which states have the most and least equitable premiums?"*

**Result:** **OK (Oklahoma)** has the lowest average premium and **GA (Georgia)** has the highest — a two-second read from the chart that would have taken much longer to spot in a raw table.

![Average premium by state — horizontal bar chart](./screenshots/image4.png)

![Average premium by state — chart detail](./screenshots/image5.png)

---


## Chapter 2 — Ingesting and Engineering Data (Bronze → Silver → Gold)

Chapter 2 moved from *querying* existing data to *building* the pipeline that feeds it — ingesting raw files, cleaning them, and progressively refining them into governed Silver and Gold tables, following the medallion architecture I'd learned conceptually in the Data Engineering fundamentals module.


### 2.1 — Manual data ingestion

I used the **Data Ingestion** UI to bring a local `vendor_data` CSV straight into a new Delta table — the fastest path for a one-off, ad-hoc dataset.

**Result:** the uploaded table had **7 columns**.

![Manually ingesting a CSV into a Delta table](./screenshots/image6.png)


### 2.2 — Incremental ingestion with `COPY INTO`

For data that arrives in batches over time (rather than a single upload), I used a **managed volume** plus the `COPY INTO` command to append new files into an existing table without duplicating what was already loaded.

```sql
COPY INTO databricks_ws_...default.vendor_data
FROM '/Volumes/databricks_ws_.../default/managed_vendor_data'
FILEFORMAT = CSV
COPY_OPTIONS ('mergeSchema' = 'true');
```

**Result:** **1,001 new rows** were written into the table — confirming `COPY INTO` only ingests what hasn't been loaded yet, which is exactly the incremental behavior a production pipeline needs.

![Using COPY INTO to append new data](./screenshots/image7.png)


### 2.3 — Cleaning raw data into the Silver layer

Real-world data is messy. I ingested a raw `employee_data_messy` table and used a `CASE` expression to backfill missing `DATE_OF_JOINING` values with `CURRENT_DATE()`, then filtered out rows with a missing `AGENT_ID` and persisted the result as a clean **Silver** table using `CREATE TABLE ... AS`.

```sql
CREATE TABLE employee_data_silver AS (
    SELECT DISTINCT *,
        CASE WHEN DATE_OF_JOINING IS NOT NULL THEN DATE_OF_JOINING
             ELSE CURRENT_DATE()
        END AS JOINING_DATE
    FROM employee_data_messy
);
```

![Cleaning the raw employee dataset](./screenshots/image8.png)

![Writing the cleaned data to the Silver table](./screenshots/image9.png)


### 2.4 — Enriching the Silver layer with a join

To support downstream OLAP-style analytics, I joined the newly ingested `insurance_data_sample` table with `employee_data_silver` on `AGENT_ID`, casting `EMP_ACCT_NUMBER` and `EMP_ROUTING_NUMBER` to `STRING` to fix inconsistent typing, and persisted the result as `agent_customer_silver`.

```sql
CREATE TABLE agent_customer_silver AS (
    SELECT ids.AGENT_ID,
           CAST(eds.EMP_ACCT_NUMBER AS STRING) AS EMP_ACCT_NUMBER,
           eds.AGENT_NAME,
           ids.PREMIUM_AMOUNT,
           CAST(eds.EMP_ROUTING_NUMBER AS STRING) AS EMP_ROUTING_NUMBER
    FROM insurance_data_sample ids
    LEFT JOIN employee_data_silver eds ON ids.AGENT_ID = eds.AGENT_ID
);
```

![Joining insurance and employee data into a Silver table](./screenshots/image10.png)

![Verifying the agent_customer_silver output](./screenshots/image11.png)


### 2.5 — Aggregating into a Gold-ready table of large claims

Finally, responding to a request from the VP of Claims Adjustments, I built a table isolating **"large claims"** (premium ≥ $100), broken down by insurance type — a Gold-layer table ready to back an executive report.

```sql
CREATE TABLE large_claims AS
SELECT COUNT(DISTINCT CUSTOMER_ID) AS customer_count,
       INSURANCE_TYPE
FROM insurance
WHERE PREMIUM_AMOUNT >= 100   -- proxy for a "large claim"
GROUP BY INSURANCE_TYPE;
```

**Result: 1,263 large claims** fell under the **Property** insurance type — the single largest segment the VP needed to review.

![Building the large_claims aggregation](./screenshots/image12.png)

![Large claims results by insurance type](./screenshots/image13.png)

---


## Chapter 3 — Analysis, Functions, and Dashboards

The final chapter tied everything together: richer analytical SQL (date functions, `CASE`-based risk scoring), turning results into visualizations, and assembling those visualizations into interactive, filterable dashboards for different audiences (agents vs. executives).


### 3.1 — Filtering claims for a targeted analysis

I narrowed the full `insurance` table down to the ten fields that mattered for a customer-service analysis, and filtered out `"Minor Loss"` incidents while keeping only `Auto`, `Motor`, and `Property` claims.

```sql
SELECT CUSTOMER_ID, PREMIUM_AMOUNT, CLAIM_AMOUNT, POLICY_EFF_DT, REPORT_DT,
       TXN_DATE_TIME, INSURANCE_TYPE, HOUSE_TYPE, INCIDENT_STATE, INCIDENT_SEVERITY
FROM insurance
WHERE INCIDENT_SEVERITY = 'Minor Loss'
  AND INSURANCE_TYPE IN ('Auto', 'Motor', 'Property');
```

**Result: 2,187 claims** matched the criteria, forming the base dataset for the rest of the chapter's analysis.

![Filtering insurance claims for customer-service analysis](./screenshots/image14.png)


### 3.2 — Engineering a churn-risk score with date functions

To flag customers at risk of churning, I used `date_diff()` to compute **CustomerMonths** (tenure) and **PaymentDays** (time between report and transaction), then bucketed customers into `High` / `Medium` / `Low` churn risk with a `CASE` expression.

```sql
SELECT *,
    date_diff(MONTH, POLICY_EFF_DT, REPORT_DT) AS CustomerMonths,
    date_diff(DAY, REPORT_DT, TXN_DATE_TIME)   AS PaymentDays,
    CASE
        WHEN CustomerMonths <= 12 AND PaymentDays > 30 THEN 'High'
        WHEN CustomerMonths > 12  AND PaymentDays <= 30 THEN 'Low'
        ELSE 'Medium'
    END AS ChurnRisk
FROM insurance
WHERE INCIDENT_SEVERITY != 'Minor Loss';
```

This is the kind of feature engineering that turns raw transactional fields into a business-usable KPI (churn risk) directly inside SQL, without needing a separate modeling step.

![Adding CustomerMonths, PaymentDays, and ChurnRisk with date functions](./screenshots/image15.png)


### 3.3 — Visualizing the churn-risk analysis

I built two visualizations on top of that query: a **line chart** of claims over time by insurance type (which showed Property claims significantly outpacing Auto), and a **bar chart** comparing claim counts and average claim amount by `ChurnRisk` bucket.

**Insight:** there were far more "Low Risk" customers than "High Risk" ones, but the *average claim amount* for High Risk customers was just as high as for Low Risk customers — meaning risk segment size and claim severity don't move together, which is a useful (and non-obvious) flag for the claims team.

![Claims over time by insurance type](./screenshots/image16.png)

![Churn risk vs. claim amounts](./screenshots/image17.png)


### 3.4 — Building an interactive Agent Analysis Dashboard

I assembled an **Agent Analysis Dashboard** joining `insurance` and `employee` data, with a text-box title, a bar chart of claim amount by insurance type, a histogram of customer age, and a **date-range filter** on `REPORT_DT`.

**Result:** filtering the dashboard to **Property** claims between **July 1 and August 30, 2020** showed a total submitted claim amount of **$5.65M**.

![Agent Analysis Dashboard with a date-range filter](./screenshots/image18.png)

![Property claims filtered by date range on the dashboard](./screenshots/image19.png)


### 3.5 — Adding a text parameter for agent-level drill-down

To make the dashboard genuinely interactive for the claims team, I added a `:agentName` **parameter** wrapped in `concat('%', :agentName, '%')` so users can filter by partial agent name directly from a dashboard widget, without touching SQL.

```sql
SELECT I.*, E.AGENT_NAME
FROM INSURANCE I
LEFT JOIN employee E ON E.AGENT_ID = I.AGENT_ID
WHERE E.AGENT_NAME LIKE concat('%', :agentName, '%');
```

**Result:** filtering for agents with **"Steve"** in their name showed that their highest claim amounts were concentrated in the **Property** insurance type.

![Using a text parameter to filter the dashboard by agent name](./screenshots/image20.png)


### 3.6 — Building an executive-facing KPI dashboard

Finally, I built a separate **Leadership KPI Dashboard** aimed at executives rather than analysts: a bar chart of total claim amount by state, a counter widget for total claim count, an `INSURANCE_TYPE` filter, and — per an executive request to see headline numbers *without* scrolling or filtering — **three Counter widgets** at the top showing total claims, total claim amount, and average premium at a glance.

![Executive KPI dashboard with counter widgets](./screenshots/image21.png)

![Executive dashboard filtered and finalized](./screenshots/image22.png)

---


## ✅ Skills demonstrated in this module

- Writing and troubleshooting ANSI SQL directly against Unity Catalog tables in the Databricks SQL Editor.
- Implementing a **Bronze → Silver → Gold** pipeline: manual and incremental (`COPY INTO`) ingestion, data cleaning with `CASE`, joins, and type casting.
- Feature engineering with date functions (`date_diff`) to build a business KPI (churn risk) directly in SQL.
- Designing bar, line, and filtered visualizations that answer specific stakeholder questions.
- Assembling interactive, parameterized dashboards for two different audiences: operational (agent-level) and executive (KPI-level).
- Connecting Databricks SQL Warehouses to external BI tools via Partner Connect.
